# Feature Store with Feast

> **This notebook is optional.** It is not required to complete the core fraud detection workshop (notebooks 0–3). You can run this notebook independently to learn about feature stores and how they integrate with ML workflows.

In this notebook, you will:
1. Understand why feature stores exist and what problems they solve
2. Set up a local Feast feature store with the fraud detection dataset
3. Define feature views, on-demand (computed) features, and feature services
4. Retrieve **historical features** for model training
5. Retrieve **online features** for real-time inference
6. Verify that Feast-served features match the original data exactly

## The Problem: Training/Serving Feature Skew

In the training notebook (`1_experiment_train.ipynb`), features are loaded like this:

```python
df = pd.read_csv('data/train.csv')
X_train = df.iloc[:, [1, 2, 4, 5, 6]].values
```

In the inference notebook (`3_rest_requests.ipynb`), the same features are manually constructed:

```python
data = [0.3111400080477545, 1.9459399775518593, 1.0, 0.0, 0.0]
```

These are **two completely disconnected code paths** for the same features. If someone changes the feature order, adds a transformation, or modifies a column in one place but not the other — the model silently produces wrong results. There is no central catalog of features, no consistency guarantee, and no way to compute derived features in one place.

A **feature store** solves this by providing a single source of truth for feature definitions that serves both training and inference.

## What is Feast?

[Feast](https://feast.dev/) (Feature Store) is an open-source feature store for machine learning. It provides:

- **Central feature registry** — one place to define, discover, and manage features
- **Historical feature retrieval** — get point-in-time correct features for training (`get_historical_features`)
- **Online feature serving** — get low-latency features for real-time inference (`get_online_features`)
- **On-demand features** — compute derived features at retrieval time, consistently across training and serving
- **Feature services** — curate different feature sets for different models or teams

**Key concepts:**

| Concept | What it is |
|---------|------------|
| **Entity** | The key used to look up features (e.g., a transaction ID, customer ID) |
| **Feature View** | A group of related features from a single data source |
| **On-Demand Feature View** | Features computed at retrieval time from other feature views |
| **Feature Service** | A curated set of features for a specific model or use case |
| **Offline Store** | Storage for historical feature data (used for training) |
| **Online Store** | Low-latency storage for the latest feature values (used for inference) |

## Install Feast

In [ ]:
!pip install feast

## Prepare the Data

The fraud detection CSV files have 8 columns of feature data, but **no entity identifier and no timestamp**. Feast strictly requires both — without them, `feast apply` will fail and feature retrieval APIs will not work.

### Why these columns are mandatory

| Column | Why Feast requires it | What happens without it |
|--------|----------------------|------------------------|
| **Entity column** (`transaction_id`) | Feast keys all feature lookups by entity. `get_online_features` needs an entity ID to know *which row* to return. `get_historical_features` joins features to your training entities by this key. | `feast apply` will fail — every `FeatureView` must reference at least one `Entity`, and that entity's `join_key` must exist in the data source. |
| **Event timestamp** (`event_timestamp`) | Feast uses timestamps for **point-in-time joins** — ensuring that `get_historical_features` only returns data that was available at each training timestamp (preventing future data leakage). It's also used during **materialization** to determine which value is the "latest" per entity for the online store. | `FileSource` requires `timestamp_field` in its constructor. Without it, Feast raises an error and cannot read the data source. |

### What we do about it

Since the original data doesn't have these columns, we **generate them synthetically** in this notebook:
- **`transaction_id`**: sequential integer (1, 2, 3, ...) — one unique ID per row
- **`event_timestamp`**: timestamps relative to the current time, 1 second apart — ensures data falls within Feast's materialization window

> **Important:** The original CSV files in `data/` are **not modified**. We load a 10,000-row subset into memory, add the two required columns, and save as a **new** Parquet file in `feast_repo/data/`. The original workshop data and flow are completely untouched. In a real-world scenario, your data pipeline would naturally produce these columns (e.g., transaction IDs from a database, timestamps from event logging).

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

FEAST_REPO = "feast_repo"
DATA_DIR = os.path.join(FEAST_REPO, "data")
SAMPLE_SIZE = 10000

Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

df = pd.read_csv('data/train.csv', nrows=SAMPLE_SIZE)
print(f"Loaded {len(df)} rows from data/train.csv (original files are NOT modified)")
print(f"Original columns ({len(df.columns)}): {list(df.columns)}")
print(f"Note: no entity ID, no timestamp — Feast cannot use this data as-is\n")

# Add the two mandatory columns
df['transaction_id'] = range(1, len(df) + 1)

now = pd.Timestamp.now(tz='UTC')
df['event_timestamp'] = now - pd.to_timedelta(range(len(df), 0, -1), unit='s')

cols = ['transaction_id', 'event_timestamp'] + [c for c in df.columns if c not in ['transaction_id', 'event_timestamp']]
df = df[cols]

# Save as a NEW parquet file in feast_repo/ (original data/ folder untouched)
parquet_path = os.path.join(DATA_DIR, "fraud_features.parquet")
df.to_parquet(parquet_path, index=False)

print(f"Saved {len(df)} rows to {parquet_path}")
print(f"New columns ({len(df.columns)}): {list(df.columns)}")
df.head()

## Configure the Feature Store

Feast uses a `feature_store.yaml` file to configure where features are stored. We use **local mode** — a file-based registry and SQLite online store — which requires no external infrastructure.

> **Sandbox note:** Workshop sandbox environments do not have permissions to deploy the Feast Operator or FeatureStore custom resource. This notebook uses local mode as a fully functional alternative. See the [Production Deployment](#production-deployment-on-openshift) section at the end for the production approach.

In [ ]:
feature_store_yaml = """project: fraud_detection
provider: local
registry: data/registry.db
online_store:
  type: sqlite
  path: data/online_store.db
offline_store:
  type: file
entity_key_serialization_version: 3
"""

with open(os.path.join(FEAST_REPO, "feature_store.yaml"), 'w') as f:
    f.write(feature_store_yaml)

print("feature_store.yaml written:")
print(feature_store_yaml)

## Define Features

Now we define the core Feast objects:

1. **Entity** — `transaction_id` is the key for feature lookups
2. **FileSource** — points to our Parquet file with the fraud detection data
3. **Feature View** — registers all 8 data columns (7 features + the fraud label) from the source
4. **On-Demand Feature View** — computes derived features at retrieval time
5. **Feature Service** — bundles stored and computed features for consumers

In [ ]:
from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource, FeatureService
from feast.on_demand_feature_view import on_demand_feature_view
from feast.types import Float64, Int64
from feast.value_type import ValueType

# 1. Entity — the key for feature lookups
transaction = Entity(
    name="transaction",
    join_keys=["transaction_id"],
    value_type=ValueType.INT64,
)

# 2. Data source — points to the Parquet file
fraud_source = FileSource(
    name="fraud_data_source",
    path=os.path.abspath(os.path.join(DATA_DIR, "fraud_features.parquet")),
    timestamp_field="event_timestamp",
)

# 3. Feature view — all data columns from the fraud detection dataset
fraud_features_view = FeatureView(
    name="fraud_features",
    entities=[transaction],
    ttl=timedelta(days=365),
    schema=[
        Field(name="distance_from_home", dtype=Float64),
        Field(name="distance_from_last_transaction", dtype=Float64),
        Field(name="ratio_to_median_purchase_price", dtype=Float64),
        Field(name="repeat_retailer", dtype=Float64),
        Field(name="used_chip", dtype=Float64),
        Field(name="used_pin_number", dtype=Float64),
        Field(name="online_order", dtype=Float64),
        Field(name="fraud", dtype=Float64),
    ],
    source=fraud_source,
    online=True,
)

print(f"Entity: {transaction.name} (join key: {transaction.join_keys})")
print(f"Source: {fraud_source.name}")
print(f"Feature View: {fraud_features_view.name} ({len(fraud_features_view.schema)} features)")

## On-Demand Features (Computed at Retrieval Time)

On-demand feature views compute derived features **at retrieval time** from other feature views. The computation runs identically for both `get_historical_features` and `get_online_features` — guaranteeing consistency between training and serving.

We define two derived features:
- **`distance_ratio`** — `distance_from_home / (distance_from_last_transaction + 0.001)` — a location anomaly signal
- **`is_high_risk`** — `1` if the transaction is online AND no PIN was used (two risk signals combined)

> **Note:** These on-demand features are for **demonstration purposes only** — they are not used by the fraud detection model trained in this workshop. They illustrate how Feast can compute derived features consistently across training and serving, eliminating the need to duplicate transformation logic.

In [ ]:
@on_demand_feature_view(
    sources=[fraud_features_view],
    schema=[
        Field(name="distance_ratio", dtype=Float64),
        Field(name="is_high_risk", dtype=Int64),
    ]
)
def fraud_risk_features(inputs: pd.DataFrame) -> pd.DataFrame:
    df = pd.DataFrame()
    df["distance_ratio"] = inputs["distance_from_home"] / (inputs["distance_from_last_transaction"] + 0.001)
    df["is_high_risk"] = ((inputs["online_order"] == 1) & (inputs["used_pin_number"] == 0)).astype(int)
    return df

print("On-demand feature view defined: fraud_risk_features")
print("  - distance_ratio: distance_from_home / (distance_from_last_transaction + 0.001)")
print("  - is_high_risk: (online_order == 1) AND (used_pin_number == 0)")

## Feature Service

A feature service bundles multiple feature views into a named set that consumers (models, pipelines) can reference. Different models can use different feature services from the same store.

In [ ]:
fraud_detection_service = FeatureService(
    name="fraud_detection_service",
    features=[fraud_features_view, fraud_risk_features],
)

print(f"Feature Service: {fraud_detection_service.name}")
print(f"  Includes: fraud_features (stored) + fraud_risk_features (on-demand)")

## Apply Feature Definitions

Applying registers all definitions (entity, feature views, feature service) with the local registry.

In [ ]:
from feast import FeatureStore

store = FeatureStore(repo_path=FEAST_REPO)

store.apply([
    transaction,
    fraud_source,
    fraud_features_view,
    fraud_risk_features,
    fraud_detection_service,
])

print("Feature definitions applied to registry.")
print(f"  Feature views: {[fv.name for fv in store.list_feature_views()]}")
print(f"  On-demand views: {[o.name for o in store.list_on_demand_feature_views()]}")
print(f"  Feature services: {[fs.name for fs in store.list_feature_services()]}")

## Materialize to Online Store

Materialization copies the latest feature values from the offline store (Parquet file) into the online store (SQLite) for low-latency lookups. We use **incremental materialization**, which only processes data that's new since the last run — the correct pattern for production use.

In [ ]:
from datetime import datetime

store.materialize_incremental(end_date=datetime.now())
print("\nMaterialization complete. Online store is ready for feature lookups.")

## Retrieve Historical Features (Training Use Case)

In a training workflow, you use `get_historical_features` to retrieve features for a set of entities at specific points in time. Feast performs a **point-in-time join** — it returns the feature values that were available at each timestamp, preventing future data from leaking into training.

Notice that features are returned as **named columns** rather than positional indices — this is more robust and self-documenting than `df.iloc[:, [1, 2, 4, 5, 6]]`.

In [ ]:
entity_df = pd.DataFrame({
    "transaction_id": list(range(1, 11)),
    "event_timestamp": [pd.Timestamp.now(tz='UTC')] * 10,
})

historical_features = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "fraud_features:distance_from_home",
        "fraud_features:distance_from_last_transaction",
        "fraud_features:ratio_to_median_purchase_price",
        "fraud_features:repeat_retailer",
        "fraud_features:used_chip",
        "fraud_features:used_pin_number",
        "fraud_features:online_order",
        "fraud_features:fraud",
    ],
).to_df()

print(f"Retrieved {len(historical_features)} rows with {len(historical_features.columns)} columns")
historical_features.sort_values('transaction_id').head(10)

## Verify: Feast Features Match Original CSV

This is the critical validation — Feast is a retrieval layer, not a transformation layer. The feature values coming out of Feast must be **identical** to what's in the original CSV.

In [ ]:
feature_cols = [
    'distance_from_home', 'distance_from_last_transaction',
    'ratio_to_median_purchase_price', 'repeat_retailer',
    'used_chip', 'used_pin_number', 'online_order', 'fraud'
]

df_csv = pd.read_csv('data/train.csv', nrows=10)
csv_values = df_csv[feature_cols].values

feast_sorted = historical_features.sort_values('transaction_id').reset_index(drop=True)
feast_values = feast_sorted[feature_cols].values

match = np.allclose(feast_values, csv_values)

print(f"Data integrity check: {'PASSED' if match else 'FAILED'}")
if match:
    print("Feast features are identical to the original CSV data.")
else:
    print("WARNING: Feast features differ from CSV! Check data preparation.")
    for col in feature_cols:
        idx = feature_cols.index(col)
        if not np.allclose(feast_values[:, idx], csv_values[:, idx]):
            print(f"  Mismatch in: {col}")

## Retrieve Online Features (Inference Use Case)

In a serving workflow, you use `get_online_features` to look up the latest feature values by entity ID. This is the low-latency path — features are served from the SQLite online store instead of scanning the full Parquet file.

In production, an inference service would call this to fetch features before sending them to the model, instead of manually constructing feature vectors.

In [ ]:
online_features = store.get_online_features(
    features=[
        "fraud_features:distance_from_last_transaction",
        "fraud_features:ratio_to_median_purchase_price",
        "fraud_features:used_chip",
        "fraud_features:used_pin_number",
        "fraud_features:online_order",
    ],
    entity_rows=[
        {"transaction_id": 1},
        {"transaction_id": 2},
        {"transaction_id": 5},
    ],
).to_dict()

print("Online features for transactions 1, 2, and 5:")
print("(These are the 5 features used by the fraud detection model)\n")
for key, values in online_features.items():
    print(f"  {key}: {values}")

## On-Demand Features in Action

On-demand features are computed at retrieval time. The same computation runs for both historical and online retrieval — no duplicated transformation logic.

> **Reminder:** These computed features (`distance_ratio`, `is_high_risk`) are for demonstration only — they are not used by the fraud detection model in this workshop.

In [ ]:
entity_df = pd.DataFrame({
    "transaction_id": list(range(1, 6)),
    "event_timestamp": [pd.Timestamp.now(tz='UTC')] * 5,
})

features_with_ondemand = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "fraud_features:distance_from_home",
        "fraud_features:distance_from_last_transaction",
        "fraud_features:online_order",
        "fraud_features:used_pin_number",
        "fraud_risk_features:distance_ratio",
        "fraud_risk_features:is_high_risk",
    ],
).to_df()

print("Historical features with on-demand computed features:")
display_cols = ['transaction_id', 'distance_from_home', 'distance_from_last_transaction',
                'distance_ratio', 'online_order', 'used_pin_number', 'is_high_risk']
features_with_ondemand.sort_values('transaction_id')[display_cols]

In [ ]:
online_with_ondemand = store.get_online_features(
    features=[
        "fraud_features:distance_from_home",
        "fraud_features:distance_from_last_transaction",
        "fraud_features:online_order",
        "fraud_features:used_pin_number",
        "fraud_risk_features:distance_ratio",
        "fraud_risk_features:is_high_risk",
    ],
    entity_rows=[{"transaction_id": 1}],
).to_dict()

print("Online features with on-demand computation for transaction 1:")
for key, values in online_with_ondemand.items():
    print(f"  {key}: {values[0]}")

print("\nThe same on-demand features are computed consistently for both historical and online retrieval.")

## Using Feast Features with the Fraud Model

This section shows the full pipeline: retrieve features from Feast → scale → predict with the ONNX model. This is the same pipeline used in the training and inference notebooks, but with Feast as the feature source instead of raw CSVs or hardcoded values.

**Prerequisite:** Run notebook `1_experiment_train.ipynb` first to create the model and scaler artifacts.

In [ ]:
import pickle

try:
    with open('artifact/scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)
    print("Loaded scaler from artifact/scaler.pkl")
except FileNotFoundError:
    print("Scaler not found. Run notebook 1_experiment_train.ipynb first.")
    scaler = None

In [ ]:
if scaler is not None:
    model_features = [
        "fraud_features:distance_from_last_transaction",
        "fraud_features:ratio_to_median_purchase_price",
        "fraud_features:used_chip",
        "fraud_features:used_pin_number",
        "fraud_features:online_order",
    ]
    model_feature_names = [
        'distance_from_last_transaction', 'ratio_to_median_purchase_price',
        'used_chip', 'used_pin_number', 'online_order'
    ]

    online_result = store.get_online_features(
        features=model_features,
        entity_rows=[{"transaction_id": 1}],
    ).to_dict()

    feature_vector = [[online_result[name][0] for name in model_feature_names]]
    scaled_vector = scaler.transform(feature_vector)

    print("Feature vector from Feast online store:")
    for name, val in zip(model_feature_names, feature_vector[0]):
        print(f"  {name}: {val}")
    print(f"\nScaled vector: {scaled_vector[0].tolist()}")
    print("\nThis scaled vector is ready to be sent to the ONNX model or KServe endpoint.")
else:
    print("Skipping — scaler not available.")

---

## Sandbox Limitation

This notebook uses Feast in **local mode** — file-based registry, SQLite online store, file offline store. This works within the workshop sandbox without any special permissions.

In a production environment, you would deploy Feast using the **Feast Operator** on OpenShift, which provides managed feature servers, PostgreSQL-backed stores, and REST/gRPC endpoints.

The sandbox environment does **not** have permissions to:
- Deploy the Feast Operator
- Create `FeatureStore` custom resources
- Provision PostgreSQL or other external stores

## Production Deployment on OpenShift

In production with Red Hat OpenShift AI, Feast is deployed using the **Feast Operator** and a **FeatureStore custom resource (CR)**.

### Example FeatureStore CR

```yaml
apiVersion: feast.dev/v1alpha1
kind: FeatureStore
metadata:
  name: fraud-detection-store
  namespace: feast
spec:
  feastProject: fraud_detection
  services:
    onlineStore:
      persistence:
        store:
          type: postgresql
          secretRef:
            name: feast-postgres-secret
    offlineStore:
      persistence:
        store:
          type: remote
    registry:
      local:
        persistence:
          store:
            type: sql
            secretRef:
              name: feast-registry-secret
```

### Key differences from local mode

| Aspect | Local (this notebook) | Production (Operator) |
|--------|----------------------|----------------------|
| Registry | File (`registry.db`) | SQL-backed (PostgreSQL) |
| Online store | SQLite | PostgreSQL |
| Offline store | File (Parquet) | Remote (Arrow Flight) |
| Feature server | In-process | Dedicated pods (REST/gRPC) |
| Auth | None | OIDC / RBAC |
| Materialization | Manual | CronJob / scheduled |

### Deployment steps (when you have operator access)

1. Install the Feast Operator via OLM or `make deploy`
2. Create infrastructure secrets (PostgreSQL credentials, registry)
3. Apply the FeatureStore CR
4. Wait for the CR to reach `Ready` state
5. Point your client `feature_store.yaml` at the remote endpoints
6. Use the same `get_historical_features` / `get_online_features` APIs — the client code does not change

## Feast Capabilities Beyond This Demo

This notebook demonstrated core Feast functionality with a local setup. In production, Feast offers additional capabilities:

- **Streaming feature ingestion** — Ingest features from Kafka or Kinesis for real-time feature updates
- **Feature monitoring** — Detect feature drift and data quality issues
- **RBAC / Permissions** — Control access to features with role-based policies (GroupBasedPolicy, NamespaceBasedPolicy)
- **Feature versioning** — Track how feature definitions change over time
- **Multi-team sharing** — Central catalog where teams discover and reuse features across projects
- **Batch materialization jobs** — Scheduled CronJobs to keep the online store fresh
- **Remote feature serving** — REST and gRPC endpoints for microservice architectures
- **Feast UI** — Web interface for browsing the feature registry, viewing feature metadata, and exploring lineage

For more information, visit the [Feast documentation](https://docs.feast.dev/).